# Machine Learning Pipeline - E-commerce

**Models:**
1. Customer Segmentation (K-Means)
2. Churn Prediction (Random Forest)
3. Product Recommendation (Popularity-based)

---

## 1. Setup & Load Data from SQLite

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle
import json
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, silhouette_score

# Config
DB_PATH = Path('data/bigdata.db')
DATA_DIR = Path('data_clean')
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

# SQLite connection
def get_db_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

# Load data from SQLite
conn = get_db_connection()
df = pd.read_sql_query('SELECT * FROM events', conn)
conn.close()

# Convert types
df['event_time'] = pd.to_datetime(df['event_time'])
df['price'] = df['price'].astype(float)
df['product_id'] = df['product_id'].astype(int)
df['user_id'] = df['user_id'].astype(int)

print(f'Loaded {len(df):,} rows from SQLite database')

## 2. Prepare Features

In [ ]:
# User-level aggregation
user_features = df.groupby('user_id').agg({
    'event_time': ['min', 'max', 'count'],
    'price': ['sum', 'mean', 'std'],
    'product_id': 'nunique',
    'event_type': lambda x: (x == 'view').sum(),
    'category_level1': 'nunique'
}).reset_index()

# Flatten columns
user_features.columns = [
    'user_id', 'first_interaction', 'last_interaction', 'total_events',
    'total_spent', 'avg_spent', 'std_spent', 'unique_products',
    'total_views', 'unique_categories'
]

# Engagement metrics
user_features['days_active'] = (
    user_features['last_interaction'] - user_features['first_interaction']
).dt.days + 1
user_features['events_per_day'] = user_features['total_events'] / user_features['days_active']
user_features['avg_session_value'] = user_features['total_spent'] / user_features['total_events']
user_features = user_features.fillna(0)

print(f'Prepared features for {len(user_features):,} users')
user_features.head()

## 3. Model 1: Customer Segmentation (K-Means)

In [ ]:
# Select features for clustering
feature_cols = ['total_events', 'total_spent', 'avg_spent', 
                'unique_products', 'unique_categories', 'events_per_day']
X = user_features[feature_cols].values

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train K-Means
N_CLUSTERS = 5
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
user_features['cluster'] = kmeans.fit_predict(X_scaled)

# Evaluate
silhouette = silhouette_score(X_scaled, user_features['cluster'])
print(f'Silhouette Score: {silhouette:.4f}')

In [ ]:
# Cluster summary
cluster_summary = user_features.groupby('cluster').agg({
    'user_id': 'count',
    'total_spent': 'mean',
    'total_events': 'mean',
    'unique_products': 'mean'
}).round(2)
cluster_summary.columns = ['count', 'avg_spent', 'avg_events', 'avg_products']
cluster_summary

In [ ]:
# Save model
with open(MODEL_DIR / 'customer_segmentation_kmeans.pkl', 'wb') as f:
    pickle.dump(kmeans, f)
with open(MODEL_DIR / 'customer_segmentation_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save to SQLite
conn = get_db_connection()
user_features[['user_id', 'cluster']].to_sql('user_clusters', conn, if_exists='replace', index=False)
conn.close()

print('Customer Segmentation model saved!')

## 4. Model 2: Churn Prediction (Random Forest)

In [ ]:
# Prepare churn labels (30 days inactive)
max_date = df['event_time'].max()
cutoff_date = max_date - pd.Timedelta(days=30)
active_users = df[df['event_time'] >= cutoff_date]['user_id'].unique()

user_activity = df.groupby('user_id').agg({
    'event_time': 'max',
    'user_session': 'nunique'
}).reset_index()
user_activity.columns = ['user_id', 'last_activity', 'total_sessions']
user_activity['churned'] = (~user_activity['user_id'].isin(active_users)).astype(int)

churn_rate = user_activity['churned'].mean()
print(f'Churn Rate: {churn_rate:.2%}')

In [ ]:
# Merge features with labels
model_data = user_features.merge(
    user_activity[['user_id', 'churned', 'total_sessions']], 
    on='user_id', how='inner'
)

# Prepare X, y
feature_cols_ml = ['total_events', 'total_spent', 'avg_spent', 'std_spent',
                   'unique_products', 'total_views', 'unique_categories',
                   'days_active', 'events_per_day', 'avg_session_value', 'total_sessions']

X = model_data[feature_cols_ml].fillna(0)
y = model_data['churned']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

In [ ]:
# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)
rf_model.fit(X_train, y_train)

# Evaluate
y_pred = rf_model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall: {recall_score(y_test, y_pred):.4f}')
print(f'F1: {f1_score(y_test, y_pred):.4f}')

In [ ]:
# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols_ml,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)
importance

In [ ]:
# Save model
with open(MODEL_DIR / 'churn_prediction_rf.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

# Save predictions to SQLite
model_data['churn_probability'] = rf_model.predict_proba(X)[:, 1]
conn = get_db_connection()
model_data[['user_id', 'churned', 'churn_probability']].to_sql('churn_predictions', conn, if_exists='replace', index=False)
conn.close()

print('Churn Prediction model saved!')

## 5. Model 3: Product Recommendation

In [ ]:
# Popularity-based recommendation
purchase_df = df[df['event_type'] == 'purchase'][['user_id', 'product_id', 'event_time']].drop_duplicates()

# Count interactions
interaction_counts = purchase_df.groupby(['user_id', 'product_id']).size().reset_index(name='count')
print(f'User-Product pairs: {len(interaction_counts):,}')

In [ ]:
# Top products per category
product_popularity = df[df['event_type'] == 'purchase'].groupby(
    ['category_level1', 'product_id']
).size().reset_index(name='popularity')

top_products = product_popularity.sort_values(
    ['category_level1', 'popularity'], ascending=[True, False]
).groupby('category_level1').head(10)

# Save to SQLite
conn = get_db_connection()
top_products.to_sql('product_recommendations', conn, if_exists='replace', index=False)
conn.close()

print('Product recommendations saved!')

## 6. Model Registry

In [ ]:
# Save model registry (MLflow-like)
registry = {
    'models': [
        {
            'name': 'customer_segmentation_kmeans',
            'type': 'clustering',
            'algorithm': 'KMeans',
            'n_clusters': N_CLUSTERS,
            'metrics': {'silhouette_score': float(silhouette)}
        },
        {
            'name': 'churn_prediction_rf',
            'type': 'classification',
            'algorithm': 'RandomForestClassifier',
            'metrics': {
                'accuracy': float(accuracy_score(y_test, y_pred)),
                'f1': float(f1_score(y_test, y_pred))
            }
        }
    ]
}

with open(MODEL_DIR / 'model_registry.json', 'w') as f:
    json.dump(registry, f, indent=2)

# Also save to SQLite
conn = get_db_connection()
pd.DataFrame(registry['models']).to_sql('model_registry', conn, if_exists='replace', index=False)
conn.close()

print('Model registry saved!')
registry